In [1]:
from src.fetch_gtopdb_targets import fetch_targets
from src.fetch_chembl_for_targets import build_datasets
from src.sort_rarg_ligands import score_ligands_by_rarg
from src.fetch_compound_info import build_compound_info
from src.fetch_target_info import build_target_info_from_compounds, build_pathway_info_from_targets
from src.fetch_clinical_trials import build_clinical_trials
from src.fetch_literature import build_literature
from src.build_ligand_summary import build_ligand_summary

from src.open_targets_known_drugs import get_known_drugs_for_target, save_known_drugs
from src.fetch_top_drugs import drugs_with_highest_phase


from pathlib import Path
import pandas as pd

In [2]:
df_lit = build_literature(
    compound_info_tsv="data/processed/compound_info.tsv",
    ligand_targets_tsv="data/processed/ligand_targets.tsv",
    out_merged="data/processed/literature_merged.tsv",
)

Literature lookup for CHEMBL4202830 ...
name_list
CHEMBL4202830
BDBM50457554
[warn] GET https://www.bindingdb.org/axis2/services/BDBRest?operation=getLigandByInchiKey&InchiKey=GXRMBYAHGICRID-UHFFFAOYSA-N → HTTP 404 (attempt 1/1)
[BindingDB] error for InChIKey=GXRMBYAHGICRID-UHFFFAOYSA-N: GET failed after 1 attempts: https://www.bindingdb.org/axis2/services/BDBRest?operation=getLigandByInchiKey&InchiKey=GXRMBYAHGICRID-UHFFFAOYSA-N


KeyboardInterrupt: 

### Initial screen of known drugs from OpenTargets
Output -> /data/processed/open_targets_know_drugs.tsv

In [ ]:
df_ot_drugs = get_known_drugs_for_target("ENSG00000172819", size=500)
save_known_drugs(df_ot_drugs)
drugs_with_highest_phase(df_ot_drugs)

### Target Receptor URLs

In [2]:
target_urls = ["https://www.guidetopharmacology.org/GRAC/ObjectDisplayForward?objectId=590&familyId=85&familyType=NHR",
               "https://www.guidetopharmacology.org/GRAC/ObjectDisplayForward?objectId=591&familyId=85&familyType=NHR",
               "https://www.guidetopharmacology.org/GRAC/ObjectDisplayForward?objectId=592&familyId=85&familyType=NHR"]

### Retrieve target information
Output -> /data/targets/gtop_targets.tsv

In [3]:
fetch_targets(target_urls, output_basename='gtop_targets.tsv')

Processing target: 590
Processing target: 591
Processing target: 592
Wrote 3 entries to C:\Users\olesk\PycharmProjects\SignalScreen\data\targets\gtop_targets.tsv


,gtop_target_id,name,type,familyIds,synonyms,gene_symbol,chembl_target_url,drugbank_target_url,systematic_nomenclature
0,590,Retinoic acid receptor-&alpha;,NHR,85,"[RAR alpha 1, RAR, RAR&alpha;, retinoic acid r...",RARA,https://www.ebi.ac.uk/chembl/explore/target/CH...,http://www.drugbank.ca/biodb/polypeptides/P10276,NR1B1
1,591,Retinoic acid receptor-&beta;,NHR,85,"[HAP, HBV-activated protein, RAR beta 2, retin...",RARB,https://www.ebi.ac.uk/chembl/explore/target/CH...,http://www.drugbank.ca/biodb/polypeptides/P10826,NR1B2
2,592,Retinoic acid receptor-&gamma;,NHR,85,"[RARC, RAR&gamma;, RAR gamma 2, retinoic acid ...",RARG,https://www.ebi.ac.uk/chembl/explore/target/CH...,http://www.drugbank.ca/biodb/polypeptides/P13631,NR1B3


### Read Targets DF if not running whole notebook

In [4]:
# 1. read the target TSV
targets_df = pd.read_csv(Path("data/targets/gtop_targets.tsv"), sep="\t")

# 2. extract the ChEMBL URL
target_urls = targets_df["chembl_target_url"].dropna().astype(str).tolist()

### Build out ligand database
Output -> data/raw/activies.tsv | data/raw/ligands.tsv

In [5]:
df_act, df_lig = build_datasets(target_urls, out_dir="data/raw")

Fetching activities for target: CHEMBL2055
 → got 1532 activity records
[WARN] Server error 500 on https://www.ebi.ac.uk/chembl/api/data/molecule/CHEMBL101666.json
[Retry] Error on attempt 1/5: Server error 500. Retrying in 1s...
[WARN] Server error 500 on https://www.ebi.ac.uk/chembl/api/data/molecule/CHEMBL101666.json
[Retry] Error on attempt 2/5: Server error 500. Retrying in 2s...
[WARN] Server error 500 on https://www.ebi.ac.uk/chembl/api/data/molecule/CHEMBL101666.json
[Retry] Error on attempt 3/5: Server error 500. Retrying in 4s...
[WARN] Server error 500 on https://www.ebi.ac.uk/chembl/api/data/molecule/CHEMBL101666.json
[Retry] Error on attempt 4/5: Server error 500. Retrying in 8s...
[WARN] Server error 500 on https://www.ebi.ac.uk/chembl/api/data/molecule/CHEMBL101666.json
[Retry] Error on attempt 5/5: Server error 500. Retrying in 16s...
Failed fetch molecule CHEMBL101666 Failed after 5 retries → https://www.ebi.ac.uk/chembl/api/data/molecule/CHEMBL101666.json
[WARN] Serve

### Score ligands based selectivity and potency
Output -> /data/processed/sorted_ligands_RARG.tsv

In [6]:
ligands = Path.cwd() / "data" / "raw" / "ligands.tsv"
activities = Path.cwd() / "data" / "raw" / "activities.tsv"

In [7]:
scored = score_ligands_by_rarg(activities, ligands)

In [8]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
scored.to_csv(Path("data/processed") / "sorted_ligands_RARG.tsv", sep="\t", index=False)

### Load scored DF and build the compound info
Output /data/processed/compound_info.tsv

In [7]:
if 'scored' not in locals():
    scored = pd.read_csv(Path.cwd() / "data" / "processed" / "sorted_ligands_RARG.tsv", sep="\t")

In [8]:
df_compound_info = build_compound_info(
    sorted_tsv="data/processed/sorted_ligands_RARG.tsv",
    raw_ligands_tsv="data/raw/ligands.tsv",
    n=25,
    out_dir="data/processed"
)

Wrote 25 compounds to data\processed\compound_info.tsv


### Find the compound target info
Output -> /data/processed/ligand_targets.tsv

In [3]:
df_tgt = build_target_info_from_compounds("data/processed/compound_info.tsv")

Fetching targets for CHEMBL4202830
[retry] GET https://www.ebi.ac.uk/chembl/api/data/activity.json failed (ReadTimeout) (attempt 1/6)
[warn] GET https://www.ebi.ac.uk/chembl/api/data/activity.json returned 500 (attempt 2/6)
[warn] GET https://www.ebi.ac.uk/chembl/api/data/activity.json returned 500 (attempt 3/6)
[warn] GET https://www.ebi.ac.uk/chembl/api/data/activity.json returned 500 (attempt 4/6)
[retry] GET https://www.ebi.ac.uk/chembl/api/data/mechanism.json failed (ReadTimeout) (attempt 1/6)
[retry] GET https://www.ebi.ac.uk/chembl/api/data/mechanism.json failed (ReadTimeout) (attempt 2/6)
[retry] GET https://www.ebi.ac.uk/chembl/api/data/mechanism.json failed (ReadTimeout) (attempt 3/6)
[retry] GET https://www.ebi.ac.uk/chembl/api/data/mechanism.json failed (ReadTimeout) (attempt 4/6)
[warn] GET https://www.ebi.ac.uk/chembl/api/data/target/CHEMBL2055.json returned 500 (attempt 1/6)
[retry] GET https://www.ebi.ac.uk/chembl/api/data/target/CHEMBL2055.json failed (ReadTimeout) (at

### Find and save pathway information related to the targets
Output -> /data/processed/ligand_pathways.tsv

In [4]:
df_pw = build_pathway_info_from_targets("data/processed/ligand_targets.tsv")

Querying Reactome for ligand CHEMBL132033 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL330998 identifiers: ['P10276', 'P10826', 'P13631', 'P19793']
Querying Reactome for ligand CHEMBL336311 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL350485 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL3707313 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL3901588 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL3907274 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL3910072 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL3926084 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL3931870 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL3936922 identifiers: ['P10276', 'P10826', 'P13631']
Querying Reactome for ligand CHEMBL39

### Find any clinical trials related to the compound
Output -> /data/processed/clinical_trials.tsv

In [5]:
df_clinical = build_clinical_trials("data/processed/compound_info.tsv")

Searching ClinicalTrials.gov for: 'CHEMBL4202830'
Searching ClinicalTrials.gov for: 'BDBM50457554'
Searching ClinicalTrials.gov for: 'CHEMBL3936922'
Searching ClinicalTrials.gov for: 'SCHEMBL381776'
Searching ClinicalTrials.gov for: 'OOTHJAQLAQMOBN-UHFFFAOYSA-N'
Searching ClinicalTrials.gov for: 'BDBM50457544'
Searching ClinicalTrials.gov for: '3''-tert-butyl-4''-diethylamino-4'-(2-hydroxyethoxy)-[1,1';3',1'']terphenyl-4-carboxylic acid'
[query_clinical_trials] Error for term '3%27%27-tert-butyl-4%27%27-diethylamino-4%27-%282-hydroxyethoxy%29-%5B1%2C1%27%3B3%27%2C1%27%27%5Dterphenyl-4-carboxylic%20acid': 400 Client Error: Bad Request for url: https://clinicaltrials.gov/api/v2/studies?query.term=3%2527%2527-tert-butyl-4%2527%2527-diethylamino-4%2527-%25282-hydroxyethoxy%2529-%255B1%252C1%2527%253B3%2527%252C1%2527%2527%255Dterphenyl-4-carboxylic%2520acid&pageSize=200
Searching ClinicalTrials.gov for: 'SCHEMBL381861'
Searching ClinicalTrials.gov for: 'CHEMBL3907274'
Searching ClinicalTri

### Find any literature related to the compound
Output -> /data/processed/literature_merged.tsv

In [2]:
df_lit = build_literature(
    compound_info_tsv="data/processed/compound_info.tsv",
    ligand_targets_tsv="data/processed/ligand_targets.tsv",
    out_merged="data/processed/literature_merged.tsv",
)

Literature lookup for CHEMBL4202830 ...
CHEMBL4202830
BDBM50457554
[warn] GET https://www.bindingdb.org/axis2/services/BDBRest?operation=getLigandByInchiKey&InchiKey=GXRMBYAHGICRID-UHFFFAOYSA-N → HTTP 404 (attempt 1/1)
[BindingDB] error for InChIKey=GXRMBYAHGICRID-UHFFFAOYSA-N: GET failed after 1 attempts: https://www.bindingdb.org/axis2/services/BDBRest?operation=getLigandByInchiKey&InchiKey=GXRMBYAHGICRID-UHFFFAOYSA-N
Literature lookup for CHEMBL3936922 ...
CHEMBL3936922
SCHEMBL381776
OOTHJAQLAQMOBN-UHFFFAOYSA-N
BDBM50457544
3''-tert-butyl-4''-diethylamino-4'-(2-hydroxyethoxy)-[1,1';3',1'']terphenyl-4-carboxylic acid
[warn] GET https://www.bindingdb.org/axis2/services/BDBRest?operation=getLigandByInchiKey&InchiKey=OOTHJAQLAQMOBN-UHFFFAOYSA-N → HTTP 404 (attempt 1/1)
[BindingDB] error for InChIKey=OOTHJAQLAQMOBN-UHFFFAOYSA-N: GET failed after 1 attempts: https://www.bindingdb.org/axis2/services/BDBRest?operation=getLigandByInchiKey&InchiKey=OOTHJAQLAQMOBN-UHFFFAOYSA-N
Literature looku

### Build and save the summary
Output -> /data/processed/ligand_deep_dive_summary

In [3]:
out = build_ligand_summary()


=== Column Audit ===
compound_info columns: ['molecule_chembl_id', 'canonical_smiles', 'standard_inchi_key', 'pubchem_cid', 'best_pubchem_name', 'all_pubchem_synonyms', 'cas_number']
ligand_targets columns: ['ligand_id', 'target_chembl_id', 'target_name', 'gene_symbol', 'moa', 'best_assay_type', 'best_affinity', 'units']
ligand_pathways columns: ['ligand_id', 'pathway_id', 'pathway_name', 'entities_found']
clinical_trials columns: ['ligand_id', 'search_term', 'nct_id', 'phase', 'status', 'indication', 'sponsor', 'results_available']
literature_merged columns: ['ligand_id', 'pmid', 'title', 'year', 'assay_type', 'assay_value', 'units', 'abstract', 'target', 'affinity', 'doi', 'source', 'link']


Summary written: (25, 18) → data/processed/ligand_deep_dive_summary.tsv

